In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, MultiHeadAttention, LayerNormalization, Dense, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv("/kaggle/input/grocery-item-patterns/grocery_cooccurrence_dataset.csv")
df['items'] = df['items'].apply(eval)  # Convert string representation to list

# Tokenize items
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['items'].apply(lambda x: ' '.join(x)))

# Create sequences
sequences = tokenizer.texts_to_sequences(df['items'].apply(lambda x: ' '.join(x)))

# Prepare input-output pairs
X = []
y = []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# Pad sequences
max_len = max(len(x) for x in X)
X = pad_sequences(X, maxlen=max_len, padding='pre')

# Convert y to categorical
num_items = len(tokenizer.word_index) + 1
y = tf.keras.utils.to_categorical(y, num_classes=num_items)

In [ ]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Transformer model parameters
embedding_dim = 64
num_heads = 4
ff_dim = 128
dropout_rate = 0.1

# Build transformer model
def build_transformer_model():
    # Input layer
    inputs = Input(shape=(max_len,))
    
    # Embedding layer
    embedding = Embedding(input_dim=num_items, output_dim=embedding_dim)(inputs)
    
    # Transformer block
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=embedding_dim)(embedding, embedding)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(embedding + attn_output)
    
    # Feed forward network
    ffn_output = Dense(ff_dim, activation="relu")(out1)
    ffn_output = Dense(embedding_dim)(ffn_output)
    ffn_output = Dropout(dropout_rate)(ffn_output)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ffn_output)
    
    # Global average pooling and output
    pooled_output = GlobalAveragePooling1D()(out2)
    outputs = Dense(num_items, activation="softmax")(pooled_output)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Create and compile model
model = build_transformer_model()
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=15,
    validation_data=(X_val, y_val)
)

# Save the model
model.save("grocery_transformer_model.h5")
tokenizer_json = tokenizer.to_json()
with open("grocery_tokenizer.json", "w") as f:
    f.write(tokenizer_json)

print("Model saved as 'grocery_transformer_model.h5'")
print("Tokenizer saved as 'grocery_tokenizer.json'")


In [ ]:
# Plot training history
import matplotlib.pyplot as plt

# Loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Prediction function
def recommend_next_items(current_items, model, tokenizer, max_len, top_k=5):
    seq = tokenizer.texts_to_sequences([current_items])[0]
    padded_seq = pad_sequences([seq], maxlen=max_len, padding='pre')
    preds = model.predict(padded_seq)[0]
    top_indices = np.argsort(preds)[-top_k:][::-1]
    recommendations = [(tokenizer.index_word[i], preds[i]) for i in top_indices if i in tokenizer.index_word]
    return recommendations

# Example usage
current_cart = ["milk 1", "cereal 1", "bananas 1"]
recommendations = recommend_next_items(current_cart, model, tokenizer, max_len)
print("Recommended items:", recommendations)